In [45]:
import pandas as pd

Units = pd.read_csv('./Units.csv')
Lines = pd.read_csv('./Lines.csv')
Links = pd.read_csv('./Links.csv')
Transformers = pd.read_csv('./Transformers.csv')
Buses = pd.read_csv('./Buses.csv') 
Loads = pd.read_csv('./Loads.csv')

file_path = "./GB_System.xlsx"

National_Data = pd.read_excel('/Users/zm348/PhD/Projects/Nature-EV/data/Inputs_prepared/GB_System_origin.xlsx', sheet_name='national_data')
Loads

,LoadID,bus_name,load_weight,RegionName
0,1,way/92419253-275,0.002569,Macclesfield
1,2,way/1190395478-220,0.000261,Strichen
2,3,way/262523325-400,0.000241,Dunbar
3,4,way/49499923-400,0.004041,Hams Hall
4,5,way/25935453-400,0.003187,Burwell Main
...,...,...,...,...
380,381,GB71-275,0.001419,Upper Boat
381,382,GB52-400,0.005545,NaN
382,383,way/87464485-400,0.004038,Rye House
383,384,way/87464485-400,0.004038,Rye House


## Initialize

In [46]:
# with pd.ExcelWriter(file_path, mode='a', engine='openpyxl') as writer:
    # Lines.to_excel(writer, sheet_name='Lines', index=False)
    # Units.to_excel(writer, sheet_name='Units', index=False)
    # Loads.to_excel(writer, sheet_name='Demands', index=False)
    # Buses.to_excel(writer, sheet_name='Buses', index=False)
    # National_Data.to_excel(writer, sheet_name='national_data', index=False)
    # Transformers.to_excel(writer, sheet_name='Transformers', index=False)
    # Links.to_excel(writer, sheet_name='Links', index=False)

In [47]:
GB_System = pd.read_excel(file_path, sheet_name=None)

## Bus ID Mapping

In [48]:
Buses['BusID'] = range(len(Buses))
Busname_to_BusID = dict(zip(Buses['bus_name'], Buses['BusID']))

## line

In [49]:
Lines.rename(columns={'LineID': 'Line_ID'}, inplace=True)
Lines['NodeIn'] = Lines['bus0'].map(Busname_to_BusID)
Lines['NodeOut'] = Lines['bus1'].map(Busname_to_BusID)
Lines.rename(columns={'capacity': 'cap'}, inplace=True)
Lines.rename(columns={'x': 'xl'}, inplace=True)

Links.rename(columns={'LinkID': 'Line_ID'}, inplace=True)
Links['NodeIn'] = Links['bus0'].map(Busname_to_BusID)
Links['NodeOut'] = Links['bus1'].map(Busname_to_BusID)
Links.rename(columns={'capacity': 'cap'}, inplace=True)
Links.rename(columns={'x': 'xl'}, inplace=True)

Transformers.rename(columns={'TransformerID': 'Line_ID'}, inplace=True)
Transformers['NodeIn'] = Transformers['bus0'].map(Busname_to_BusID)
Transformers['NodeOut'] = Transformers['bus1'].map(Busname_to_BusID)
Transformers.rename(columns={'s_nom': 'cap'}, inplace=True)

line = pd.concat([Lines, Links, Transformers], ignore_index=True)
line = line[['Line_ID', 'NodeIn', 'NodeOut', 'cap', 'xl']]
line['Line_ID'] = range(len(line))
line

,Line_ID,NodeIn,NodeOut,cap,xl
0,0,175,419.0,921.668,14.577213
1,1,99,88.0,3574.953,3.447219
2,2,118,35.0,1843.335,0.310518
3,3,474,128.0,3574.953,3.492045
4,4,139,74.0,1474.668,7.949573
...,...,...,...,...,...
735,735,372,NaN,NaN,NaN
736,736,376,216.0,NaN,NaN
737,737,386,NaN,NaN,NaN
738,738,388,315.0,NaN,NaN


In [50]:
line.loc[line['NodeOut'].isna(), 'NodeOut'] = line.loc[line['NodeOut'].isna(), 'NodeIn']
line['cap'] = line['cap'].fillna(line['cap'].mean())
line['xl'] = line['xl'].fillna(line['xl'].mean())

# p.u.
Xbase = 1
line['xl'] = line['xl']/Xbase

## Loads Mapping

In [51]:
Loads_merged = Loads.groupby('bus_name', as_index=False)['load_weight'].sum()
BusID_to_Loads = dict(zip(Loads_merged['bus_name'].map(Busname_to_BusID), Loads_merged['load_weight']))

## dem

In [52]:
dem = pd.DataFrame()
dem['Bus_ID'] = Buses['BusID']
dem['No'] = range(1, len(dem)+1)
dem['base'] = dem['Bus_ID'].map(BusID_to_Loads)
dem['base'] = dem['base'].fillna(0)
dem

,Bus_ID,No,base
0,0,1,0.000000
1,1,2,0.000426
2,2,3,0.000000
3,3,4,0.000000
4,4,5,0.000407
...,...,...,...
539,539,540,0.000000
540,540,541,0.000000
541,541,542,0.000000
542,542,543,0.000000


In [53]:
mask = National_Data.iloc[:, 0] == 'Demand'
demand_dict = National_Data.loc[mask].iloc[0, 1:].to_dict()

for t_col in demand_dict.keys():
    dem[t_col] = dem['base'] * demand_dict[t_col]

## dg

In [54]:
Units = Units[Units['Status'] == 'operating']
Units['Bus_ID'] = Units['Bus name'].map(Busname_to_BusID)


In [55]:
Units = Units[['Technology', 'Bus_ID', 'capacity']]
Units = Units.groupby(['Technology', 'Bus_ID'], as_index=False)['capacity'].sum()
dg = Units.sort_values(by=['Bus_ID', 'Technology']).reset_index(drop=True)
dg['NO'] = range(1, len(dg) + 1)
dg.rename(columns={'Technology': 'Type', 'capacity': 'Pmax'}, inplace=True)

In [56]:
tech_costs = {
    'biomass': 98,
    'solar': 44,
    'offwind': 65,
    'onwind': 45,
    'nuclear': 92,
    'hydro': 90,
    'coal':100,
    'oil':130,
    'gas':70
}


carbon_intensity = {
    'biomass': 0,    
    'solar': 0,      
    'offwind': 0,    
    'onwind': 0,     
    'nuclear': 0,    
    'hydro': 0,      
    'coal': 820,     
    'oil': 650,      
    'gas': 490       
}

dg['lambdaG'] = dg['Type'].map(tech_costs)
dg['Carbon'] = dg['Type'].map(carbon_intensity)




## res

In [57]:
res = dg.copy()
res['base'] = res.groupby('Type')['Pmax'].transform(lambda x: x / x.sum())
res = res[['Bus_ID','base','Type']]
res.rename(columns={'Type': 'NO'}, inplace=True)
res = res[res['NO'].isin(['onwind', 'offwind', 'solar'])]
res

,Bus_ID,base,NO
0,0,0.000726,offwind
1,0,0.004365,onwind
3,1,0.005675,onwind
4,1,0.000506,solar
5,4,0.011950,onwind
...,...,...,...
503,486,0.000890,solar
505,488,0.014787,onwind
508,489,0.008997,offwind
510,490,0.033169,onwind


In [58]:
mask_wind = National_Data.iloc[:, 0] == 'Wind'
mask_solar = National_Data.iloc[:, 0] == 'Solar'
demand_dict_wind = National_Data.loc[mask_wind].iloc[0, 1:].to_dict()
demand_dict_solar = National_Data.loc[mask_solar].iloc[0, 1:].to_dict()

for t_col in demand_dict_wind.keys():  # shared keys
    res[t_col] = 0.0
    res.loc[res['NO'].isin(['onwind', 'offwind']), t_col] = \
        res['base'] * demand_dict_wind[t_col] / 2 # Divide by 2 for onwind and offwind
    res.loc[res['NO'] == 'solar', t_col] = \
        res['base'] * demand_dict_solar[t_col]


# total_T5_solar = res.loc[res['NO'] == 'solar', 'T5'].sum()
# print(total_T0_solar)
# total_T0_wind = res.loc[res['NO'] == 'onwind', 'T0'].sum()
# print(total_T0_wind)
res

,Bus_ID,base,NO,T0,T1,T2,T3,T4,T5,T6,...,T38,T39,T40,T41,T42,T43,T44,T45,T46,T47
0,0,0.000726,offwind,2.534471,2.518039,2.495491,2.491467,2.478761,2.464770e+00,2.456664,...,2.663703,2.666953,2.671529,2.669253,2.665190,2.654231,2.632054,2.603780,2.590145,2.557048
1,0,0.004365,onwind,15.247701,15.148841,15.013188,14.988983,14.912540,1.482837e+01,14.779603,...,16.025175,16.044728,16.072258,16.058565,16.034120,15.968190,15.834774,15.664673,15.582640,15.383526
3,1,0.005675,onwind,19.824626,19.696091,19.519720,19.488249,19.388860,1.927943e+01,19.216019,...,20.835476,20.860899,20.896692,20.878889,20.847107,20.761386,20.587922,20.366762,20.260104,20.001223
4,1,0.000506,solar,0.000000,0.000000,0.000000,0.000000,0.000000,4.417508e-10,0.000733,...,0.003275,0.000009,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,4,0.011950,onwind,41.741561,41.470925,41.099568,41.033305,40.824038,4.059362e+01,40.460113,...,43.869947,43.923475,43.998840,43.961354,43.894436,43.713947,43.348712,42.883051,42.658479,42.113393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503,486,0.000890,solar,0.000000,0.000000,0.000000,0.000000,0.000000,7.774814e-10,0.001290,...,0.005764,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
505,488,0.014787,onwind,51.653874,51.318971,50.859428,50.777429,50.518468,5.023334e+01,50.068122,...,54.287686,54.353925,54.447187,54.400799,54.317990,54.094640,53.642673,53.066432,52.788531,52.114004
508,489,0.008997,offwind,31.427443,31.223680,30.944083,30.894193,30.736635,3.056315e+01,30.462634,...,33.029916,33.070218,33.126960,33.098737,33.048354,32.912463,32.637475,32.286876,32.117795,31.707397
510,490,0.033169,onwind,115.861601,115.110400,114.079629,113.895702,113.314843,1.126753e+02,112.304699,...,121.769340,121.917918,122.127107,122.023058,121.837313,121.336331,120.322553,119.030022,118.406679,116.893690


# Write datas into xlsx

In [59]:
with pd.ExcelWriter('GB_System.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    line.to_excel(writer, sheet_name='line', index=False)
    dem.to_excel(writer, sheet_name='dem', index=False)
    res.to_excel(writer, sheet_name='res', index=False)
    dg.to_excel(writer, sheet_name='dg', index=False)

